## 1. The PyReason program (for reference)

```python
# examples/minimal_example.py  (lab-v2/pyreason @ 46ea97c)
g = nx.DiGraph()
g.add_nodes_from(['John', 'Mary', 'Justin', 'Dog', 'Cat'])
g.add_edge('Justin', 'Mary', Friends=1)
g.add_edge('John',   'Mary', Friends=1)
g.add_edge('John',   'Justin', Friends=1)
g.add_edge('Mary',   'Cat', owns=1)
g.add_edge('Justin', 'Cat', owns=1)
g.add_edge('Justin', 'Dog', owns=1)
g.add_edge('John',   'Dog', owns=1)

# Immediate rule (delta_t = 0): whole cascade resolves within one timestep via the fixpoint.
pr.add_rule(pr.Rule('popular(x) <- popular(y), Friends(x,y), owns(y,z), owns(x,z)', 'popular_rule'))
pr.add_fact(pr.Fact('popular(Mary)', 'popular_fact', 0, 0))
interpretation = pr.reason(timesteps=0)
```

In [1]:
# Use the live workspace source, bypassing any stale editable install.
import sys, pathlib
here = pathlib.Path.cwd()
for cand in (here, *here.parents):
    if (cand / 'src' / 'srdatalog' / 'hir' / '__init__.py').exists():
        sys.path.insert(0, str(cand / 'src'))
        print('srdatalog src:', cand / 'src')
        break

from srdatalog.dsl import Var, Relation, Program

srdatalog src: /home/stargazermiao/workspace/srdatalog-python/src


## 2. The same program in the srdatalog DSL

The PyReason graph edges become EDB relations; the rule maps one-to-one onto `<=` / `&`. (srdatalog compiles this to GPU C++; here we just build the program object — its least model is the popular reachable set we compare against below.)

In [2]:
x, y, z = Var('x'), Var('y'), Var('z')
Friends = Relation('Friends', 2)   # edge attribute Friends=1
owns    = Relation('owns', 2)      # edge attribute owns=1
popular = Relation('popular', 1)   # IDB + the seed fact popular(Mary)

prog = Program(rules=[
    (popular(x) <= popular(y) & Friends(x, y) & owns(y, z) & owns(x, z)).named('popular_rule'),
])

print('relations:', [(r.name, r.arity) for r in prog.relations])
print('rule:     ', prog.rules[0].name, '|', prog.rules[0])

relations: [('popular', 1), ('Friends', 2), ('owns', 2)]
rule:      popular_rule | Rule(heads=(Atom(rel='popular', args=(ClauseArg(kind=<ArgKind.LVAR: 'var'>, var_name='x', const_value=None, const_cpp_expr=None, cpp_code=None),), prov=Provenance(kind=<ProvenanceKind.USER: 'pkUser'>, parent_rule='', derived_from='', transform_pass='')),), body=(Atom(rel='popular', args=(ClauseArg(kind=<ArgKind.LVAR: 'var'>, var_name='y', const_value=None, const_cpp_expr=None, cpp_code=None),), prov=Provenance(kind=<ProvenanceKind.USER: 'pkUser'>, parent_rule='', derived_from='', transform_pass='')), Atom(rel='Friends', args=(ClauseArg(kind=<ArgKind.LVAR: 'var'>, var_name='x', const_value=None, const_cpp_expr=None, cpp_code=None), ClauseArg(kind=<ArgKind.LVAR: 'var'>, var_name='y', const_value=None, const_cpp_expr=None, cpp_code=None)), prov=Provenance(kind=<ProvenanceKind.USER: 'pkUser'>, parent_rule='', derived_from='', transform_pass='')), Atom(rel='owns', args=(ClauseArg(kind=<ArgKind.LVAR: 'var'>, v

## 3. Run PyReason live and compare

We actually run `minimal_example` through PyReason here and check its `t=0` interpretation equals the least model of the Section-2 srdatalog program (`{Mary, Justin, John}`, all bounds `[1,1]`).

PyReason isn't pip-installed in this environment — it imports only from its repo root — so the next cell locates a sibling `pyreason/` checkout and puts it on `sys.path`. (First run pays a numba JIT-compile cost of a minute or two.) If no checkout is found, the cell prints a note and the comparison is skipped so the notebook still completes standalone.

In [3]:
# --- Run PyReason's minimal_example live ---
import sys, pathlib

def _load_pyreason():
    try:
        import pyreason as pr
        return pr
    except ImportError:
        pass
    here = pathlib.Path.cwd()
    for anc in (here, *here.parents):
        cand = anc.parent / 'pyreason'        # sibling checkout under the workspace
        if (cand / 'pyreason' / '__init__.py').exists():
            sys.path.insert(0, str(cand))
            import pyreason as pr
            return pr
    return None

pr = _load_pyreason()
pyreason_result = None
if pr is None:
    print('pyreason checkout not found - skipping live run; comparison falls back to known t=0 set.')
else:
    import networkx as nx
    pr.reset(); pr.reset_rules(); pr.reset_settings()
    pr.settings.verbose = False
    g = nx.DiGraph()
    g.add_nodes_from(['John', 'Mary', 'Justin', 'Dog', 'Cat'])
    g.add_edge('Justin', 'Mary', Friends=1); g.add_edge('John', 'Mary', Friends=1); g.add_edge('John', 'Justin', Friends=1)
    g.add_edge('Mary', 'Cat', owns=1); g.add_edge('Justin', 'Cat', owns=1)
    g.add_edge('Justin', 'Dog', owns=1); g.add_edge('John', 'Dog', owns=1)
    pr.load_graph(g)
    pr.add_rule(pr.Rule('popular(x) <- popular(y), Friends(x,y), owns(y,z), owns(x,z)', 'popular_rule'))
    pr.add_fact(pr.Fact('popular(Mary)', 'popular_fact', 0, 0))
    interp = pr.reason(timesteps=0)
    dfs = pr.filter_and_sort_nodes(interp, ['popular'])
    print('PyReason timesteps returned:', len(dfs))
    pyreason_result = {r['component']: tuple(r['popular']) for _, r in dfs[0].iterrows()}
    print('PyReason t=0 popular:', pyreason_result)

Added  0 graph-attribute node facts and  7 graph_attribute edge facts.


PyReason timesteps returned: 1
PyReason t=0 popular: {'Mary': (1.0, 1.0), 'Justin': (1.0, 1.0), 'John': (1.0, 1.0)}


In [4]:
# Least model of the Section-2 srdatalog program (the popular reachable set):
# Mary (seed) -> Justin (shares Cat with Mary) -> John (shares Dog with Justin).
srdatalog_least_model = {'Mary', 'Justin', 'John'}

if pyreason_result is not None:
    pyreason_set = set(pyreason_result)
    all_boolean = all(b == (1.0, 1.0) for b in pyreason_result.values())
    assert pyreason_set == srdatalog_least_model, (pyreason_set, srdatalog_least_model)
    assert all_boolean, pyreason_result
    print('PyReason t=0 popular         :', sorted(pyreason_set), '(all bounds [1,1])')
    print('srdatalog program least model:', sorted(srdatalog_least_model))
    print('\nPASS - identical node sets; PyReason bounds collapse to Boolean [1,1].')
else:
    print('PyReason unavailable; expected least model =', sorted(srdatalog_least_model))

print("\nminimal_example is fully covered by srdatalog's plain-Datalog semantics.")

PyReason t=0 popular         : ['John', 'Justin', 'Mary'] (all bounds [1,1])
srdatalog program least model: ['John', 'Justin', 'Mary']

PASS - identical node sets; PyReason bounds collapse to Boolean [1,1].

minimal_example is fully covered by srdatalog's plain-Datalog semantics.


## 4. Where does the `<-N` delay change the fixpoint?

`minimal_example` uses `<-` (delay 0). PyReason's general form is `<-N`: "if the body holds at time `t`, the head holds at `t+N`". Does the delay value change the *answer* (the fixpoint), or only the schedule by which it is reached?

| example (real graph/data) | character | delay changes fixpoint? |
|---|---|---|
| `basic_tutorial_ex` | positive recursion (popular) | **no** — union `{John, Justin, Mary}` for native/zero/plus1 |
| `infer_edges_ex` | positive + `infer_edges` (airport graph) | **no** — identical 19-atom union (10 nodes + 9 `isConnectedTo` edges) |
| `load_rules_facts_from_file` | positive (eligible / under_department) | **no** — identical union |
| `closed_world_pred_ex` | **negation + closed-world** | **YES** — native/plus1 derive `inconsistent@cb_2`; `zero` does not |
| `advanced_graph_ex`, `text` | positive | not runnable — fact-parse error in this checkout (delay-independent) |
| `temporal_classifier_ex`, `weather_temporal_classifier_ex` | positive, but a conjunction over a finite-window fact | needs torch; not run here — structurally delay-sensitive (a delayed atom must coincide in time with a short-lived one) |

In [5]:
# Live in-kernel proof on two real programs (each run fully reset -> no state bleed).
# Fixpoint = union over all timesteps, read via filter_and_sort_nodes (the reliable API;
# get_dict() is a per-timestep change-log and under-reports persistent atoms).
_ARR = {'native': '<-1', 'zero': '<-', 'plus1': '<-2'}

def _union_nodes(interp, labels, qualify):
    u = set()
    for lab in labels:
        for df in pr.filter_and_sort_nodes(interp, [lab]):
            for comp in df['component'].tolist():
                u.add(f'{lab}@{comp}' if qualify else comp)
    return sorted(u)

def popular_union(mode, T=20):
    pr.reset(); pr.reset_rules(); pr.reset_settings(); pr.settings.verbose = False
    g = nx.DiGraph(); g.add_nodes_from(['John', 'Mary', 'Justin', 'Dog', 'Cat'])
    g.add_edge('Justin', 'Mary', Friends=1); g.add_edge('John', 'Mary', Friends=1); g.add_edge('John', 'Justin', Friends=1)
    g.add_edge('Mary', 'Cat', owns=1); g.add_edge('Justin', 'Cat', owns=1); g.add_edge('Justin', 'Dog', owns=1); g.add_edge('John', 'Dog', owns=1)
    pr.load_graph(g)
    pr.add_rule(pr.Rule(f'popular(x) {_ARR[mode]} popular(y), Friends(x,y), owns(y,z), owns(x,z)', 'r'))
    pr.add_fact(pr.Fact('popular(Mary)', 'f', 0, 20))      # persistent seed
    return _union_nodes(pr.reason(timesteps=T), ['popular'], qualify=False)

def cw_union(mode, T=20):  # faithful closed_world_pred_ex.py: all three rules
    pr.reset(); pr.reset_rules(); pr.reset_settings()
    pr.settings.verbose = False; pr.settings.inconsistency_check = True
    g = nx.DiGraph(); g.add_nodes_from(['cb_1', 'cb_2', 'l1', 'l2'])
    g.add_edge('cb_1', 'cb_2', stepFrom=1); g.add_edge('cb_1', 'l1', hasLabel=1); g.add_edge('cb_2', 'l2', hasLabel=1)
    pr.load_graph(g); pr.add_closed_world_predicate('hackerControl')
    pr.add_fact(pr.Fact('stepFrom(cb_1, cb_2)', 'sf', 0, 1)); pr.add_fact(pr.Fact('hackerControl(cb_1)', 'hc', 0, 0))
    pr.add_rule(pr.Rule(f'future(Y) {_ARR[mode]} stepFrom(X,Y), hackerControl(X)', 'fr'))
    pr.add_rule(pr.Rule(f'hackerControl(Y) {_ARR[mode]} hackerControl(X), hasLabel(X,L1), hasLabel(Y,L2), cond(L1, L2), stepFrom(X,Y)', 'hr'))
    pr.add_rule(pr.Rule('inconsistent(Y) <- future(Y), ~hackerControl(Y), ~hackerControl(X)', 'ir'))
    return _union_nodes(pr.reason(timesteps=T), ['future', 'inconsistent'], qualify=True)

if pr is not None:
    pop = {m: popular_union(m) for m in ('native', 'zero', 'plus1')}
    cw  = {m: cw_union(m)      for m in ('native', 'zero', 'plus1')}

    print('popular (positive)         native={} zero={} plus1={}'.format(pop['native'], pop['zero'], pop['plus1']))
    print('  delay-invariant?', pop['native'] == pop['zero'] == pop['plus1'], '\n')
    print('closed_world (negation/CW) native = {}'.format(cw['native']))
    print('                           zero   = {}'.format(cw['zero']))
    print('                           plus1  = {}'.format(cw['plus1']))
    print('  delay-invariant?', cw['native'] == cw['zero'] == cw['plus1'])

    assert pop['native'] == pop['zero'] == pop['plus1'] == ['John', 'Justin', 'Mary']
    assert 'inconsistent@cb_2' in cw['native'] and 'inconsistent@cb_2' not in cw['zero']
    print('\nCONFIRMED on real programs: positive => delay-invariant fixpoint;')
    print('negation/closed-world => fixpoint depends on delay (zero loses inconsistent@cb_2).')
else:
    print('pyreason unavailable - skipping.')

Added  0 graph-attribute node facts and  7 graph_attribute edge facts.
Added  0 graph-attribute node facts and  7 graph_attribute edge facts.
Added  0 graph-attribute node facts and  7 graph_attribute edge facts.
Added  0 graph-attribute node facts and  3 graph_attribute edge facts.


Added  0 graph-attribute node facts and  3 graph_attribute edge facts.
Added  0 graph-attribute node facts and  3 graph_attribute edge facts.
popular (positive)         native=['John', 'Justin', 'Mary'] zero=['John', 'Justin', 'Mary'] plus1=['John', 'Justin', 'Mary']
  delay-invariant? True 

closed_world (negation/CW) native = ['future@cb_2', 'inconsistent@cb_2']
                           zero   = ['future@cb_2']
                           plus1  = ['future@cb_2', 'inconsistent@cb_2']
  delay-invariant? False

CONFIRMED on real programs: positive => delay-invariant fixpoint;
negation/closed-world => fixpoint depends on delay (zero loses inconsistent@cb_2).
